# 22b. Các khối của tập test và đoạn dẫn vào của từng đợt tấn công (GĐ J, task 8.2), CPU
Chia `blocked` 10 khối nên tập test gồm các khối thời gian xen kẽ với train/val. Với mỗi đợt tấn công của `realtime_eval.json` (notebook 20): cửa sổ đầu của đợt có phải
đầu khối test không, và trước đó trong cùng khối test có bao nhiêu cửa sổ (đoạn dẫn vào, không có node tấn công nào vì mọi cửa sổ có node tấn công đều thuộc một đợt).
Chỉ đọc `split.json` và `realtime_eval.json`, không chạy mô hình.

In [ ]:
import glob, json, os
from datetime import datetime, timezone


def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern}"
    return hits[0]


def hms(t):
    return datetime.fromtimestamp(t, tz=timezone.utc).strftime("%d/%m %H:%M:%S")


meta = json.load(open(find("nf_ton_iot_v3/split.json"), encoding="utf-8"))
ev = json.load(open(find("gnn-iot-ids-20-realtime/realtime_eval.json"), encoding="utf-8"))
tw = sorted(int(w) for w in meta["test"]["windows"])
blocks, cur = [], [tw[0]]
for w in tw[1:]:
    if w - cur[-1] > 100:                      # test nằm xen kẽ train/val: giữa hai khối cách nhau hàng nghìn cửa sổ
        blocks.append(cur)
        cur = []
    cur.append(w)
blocks.append(cur)
print(f"{len(tw)} cửa sổ test, {len(blocks)} khối")
for i, b in enumerate(blocks):
    print(f"  khối {i}: {hms(b[0] * 5)} → {hms((b[-1] + 1) * 5)} UTC, {len(b)} cửa sổ có flow (bước ≤ 100)")
rows = []
for e in ev["detection"]["episodes"]:
    fw = e["first_window"]
    b = next(bl for bl in blocks if bl[0] <= fw <= bl[-1])
    lead = sum(1 for w in b if w < fw)
    rows.append(dict(cls=e["class"], start=hms(e["start_ts"]), n_windows=e["n_windows"], detected=e["detected"], detection_s=e["detection_s"],
                     block_start=hms(b[0] * 5), lead_windows=lead, lead_s=lead * 5, first_window=fw))
print()
for r in rows:
    print(f"{r['cls']:<20} {r['start']}  {r['n_windows']:>5} cửa sổ  phát hiện={str(r['detected']):<5} t={r['detection_s']}  | khối bắt đầu {r['block_start']}, đoạn dẫn vào trong test: {r['lead_windows']} cửa sổ ({r['lead_s']} s)")
os.makedirs("/kaggle/working", exist_ok=True)
json.dump({"blocks": [{"start": b[0] * 5, "end": (b[-1] + 1) * 5, "n_windows": len(b)} for b in blocks], "episodes": rows}, open("/kaggle/working/test_blocks.json", "w"), indent=1)